<a href="https://colab.research.google.com/github/aiunderstand/USN-CS4140-EmbeddedAI//blob/master/examples/hello_mnist/train/mnist_model_tensorflow_lite_micro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST for LiteRT / TensorFlow Lite Micro

This notebook trains a small MNIST digit-recognition model, quantizes it to **INT8**, and exports it as a C array ready to flash onto a microcontroller.

We demonstrate two quantization strategies:
- **Path A – Post-Training Quantization (PTQ):** Quantize after training using a small representative dataset.
- **Path B – Quantization-Aware Training (QAT):** Fine-tune the model with simulated quantization to recover accuracy.

The exported `.tflite` files can be deployed with [LiteRT for Microcontrollers](https://ai.google.dev/edge/litert/microcontrollers/overview) (formerly TensorFlow Lite Micro), the standard runtime for on-device ML on embedded hardware.

# Installation

Pin TensorFlow to Colab's pre-installed version (2.19) and install `tf-keras` (legacy Keras 2) plus `tensorflow-model-optimization` for QAT support. TensorFlow 2.16+ ships Keras 3 by default, but `tensorflow-model-optimization` requires the legacy Keras 2 API (see [keras.io/getting_started](https://keras.io/getting_started/) and [keras-team/keras#20319](https://github.com/keras-team/keras/discussions/20319)). The `tf-keras` package provides this backward-compatible API.

In [ ]:
# Install the QAT toolkit.  We pin TF to Colab's pre-installed version
# to avoid numpy / dependency conflicts.
# tf-keras provides the legacy Keras 2 API needed by tensorflow-model-optimization for QAT.
# (TF 2.16+ ships Keras 3 by default, which is NOT compatible with QAT.)
# ⚠️  Restart the runtime after this cell (Runtime → Restart runtime).
!pip install "tensorflow==2.19.0" tf-keras tensorflow-model-optimization ai-edge-litert -q

> ⚠️ **After running the cell above, restart the runtime** (`Runtime → Restart runtime`) before continuing. This ensures the correct package versions are loaded.

# Python Modules

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import matplotlib.pyplot as plt
import random

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers
import tensorflow_model_optimization as tfmot
from ai_edge_litert import interpreter as litert

print("TensorFlow:", tf.__version__)
print("Keras (legacy):", keras.__version__)

# Loading MNIST Data

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 60,000 training images and 10,000 test images, each 28×28 pixels
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test  shape:", x_test.shape)
print("y_test  shape:", y_test.shape)

In [ ]:
fig = plt.figure(figsize=(9, 9))

for i in range(9):
    plt.subplot(3, 3, i + 1)
    num = random.randint(0, len(x_train) - 1)
    plt.imshow(x_train[num], cmap="gray", interpolation=None)
    plt.title(y_train[num])

plt.tight_layout()

# Create Model

A compact model well-suited for microcontrollers: `Flatten → Dense(64, relu) → Dense(10, softmax)`. The `Flatten` layer converts each 28×28 image to a 784-element vector, so we keep the raw 2-D shape throughout data loading.

In [ ]:
def create_model():
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28)),   # 28×28 → 784
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )
    return model

model = create_model()
model.summary()

# Prepare Data

Normalize pixel values to `[0, 1]`. We keep images in their original `(28, 28)` shape — the `Flatten` layer inside the model handles the flattening automatically.

In [ ]:
# Normalize to float32 in [0, 1] — keep the 2-D shape for the Flatten layer
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

print("x_train shape:", x_train.shape)
print("x_test  shape:", x_test.shape)

# Train and Save Model

`validation_split=0.1` holds out 10 % of the training data so we can monitor overfitting. After training we save in the modern `.keras` format.

In [ ]:
model.fit(
    x=x_train,
    y=y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=1,
)

scores = model.evaluate(x_test, y_test, verbose=2)
print("Test Loss:    ", scores[0])
print("Test Accuracy:", scores[1])

model.save("mnist_model.keras")

# Path A — Full INT8 Post-Training Quantization (PTQ)

PTQ is the quickest way to shrink a model. After training, we run ~100 representative input samples through the converter so it can compute the optimal int8 scale/zero-point for every activation. No extra training is required.

Key converter settings:
- `tf.lite.Optimize.DEFAULT` — enable full quantization
- `TFLITE_BUILTINS_INT8` — use only INT8 kernels
- `inference_input_type` / `inference_output_type = tf.int8` — MCUs send raw int8 bytes

In [ ]:
# Use 100 samples from the training set as the representative dataset
x_rep = x_train[:100].reshape(-1, 28, 28)   # keep 2-D shape

def representative_dataset():
    for sample in x_rep:
        yield [sample[np.newaxis, :, :]]     # shape (1, 28, 28)

converter_ptq = tf.lite.TFLiteConverter.from_keras_model(model)
converter_ptq.optimizations = [tf.lite.Optimize.DEFAULT]
converter_ptq.representative_dataset = representative_dataset
converter_ptq.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_ptq.inference_input_type  = tf.int8
converter_ptq.inference_output_type = tf.int8

tflite_ptq = converter_ptq.convert()
open("model_ptq_int8.tflite", "wb").write(tflite_ptq)
print("PTQ INT8 model saved — size:", len(tflite_ptq), "bytes")

# Path B — Quantization-Aware Training (QAT)

QAT inserts fake quantization nodes into the graph during fine-tuning, so the model *learns* to be robust to int8 rounding. This usually closes the small accuracy gap that PTQ can leave on challenging tasks.

We use the legacy Keras 2 API (`tf-keras`) because `tensorflow-model-optimization` does not yet support Keras 3 (see [keras-team/keras#20319](https://github.com/keras-team/keras/discussions/20319)).

Steps:
1. Convert the Sequential model to a Functional model (required by `tfmot.quantization.keras.quantize_model()`).
2. `tfmot.quantization.keras.quantize_model()` wraps every layer with fake-quant ops.
3. Fine-tune for a few epochs (recompile first — the wrapper changes the optimizer state).
4. Convert exactly like PTQ.

In [ ]:
# 1. Wrap the trained model with QAT fake-quantization
# Note: tfmot.quantization.keras.quantize_model() requires a Functional model.
functional_model = keras.Model(inputs=model.input, outputs=model.output)
q_aware_model = tfmot.quantization.keras.quantize_model(functional_model)

q_aware_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
q_aware_model.summary()

In [ ]:
# 2. Fine-tune for 5 epochs
q_aware_model.fit(
    x=x_train,
    y=y_train,
    batch_size=128,
    epochs=5,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
# 3. Convert the QAT model to INT8 TFLite
converter_qat = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter_qat.optimizations = [tf.lite.Optimize.DEFAULT]
converter_qat.representative_dataset = representative_dataset
converter_qat.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_qat.inference_input_type  = tf.int8
converter_qat.inference_output_type = tf.int8

tflite_qat = converter_qat.convert()
open("model_qat_int8.tflite", "wb").write(tflite_qat)
print("QAT INT8 model saved — size:", len(tflite_qat), "bytes")

# Model Size Comparison

INT8 models store weights as 8-bit integers instead of 32-bit floats — roughly **4× smaller**. On a microcontroller with only tens of kilobytes of flash, this difference matters enormously.

In [ ]:
# Save the float model as TFLite for a fair comparison
converter_float = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter_float.convert()
open("model_float32.tflite", "wb").write(tflite_float)

sizes = {
    "Float32 TFLite": os.path.getsize("model_float32.tflite"),
    "PTQ INT8      ": os.path.getsize("model_ptq_int8.tflite"),
    "QAT INT8      ": os.path.getsize("model_qat_int8.tflite"),
}

print(f"{'Model':<20} {'Size (bytes)':>14} {'Size (KB)':>10}")
print("-" * 46)
for name, size in sizes.items():
    print(f"{name:<20} {size:>14,} {size/1024:>9.1f}")

# Test TFLite Models

We test **sample-by-sample** — exactly how inference runs on a microcontroller. For INT8 models the input must be scaled from `float32` to `int8` using the quantization parameters reported by the interpreter.

In [ ]:
def evaluate_tflite(model_path, x, y_true, max_samples=1000):
    """Run TFLite inference sample-by-sample and return accuracy."""
    interpreter = litert.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype = input_details[0]["dtype"]
    is_int8 = (input_dtype == np.int8)

    if is_int8:
        scale, zero_point = input_details[0]["quantization"]

    correct = 0
    for i in range(min(max_samples, len(x))):
        sample = x[i:i+1]                       # shape (1, 28, 28)

        if is_int8:
            sample = (sample / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        pred = np.argmax(output[0])
        if pred == y_true[i]:
            correct += 1

    return correct / min(max_samples, len(x))

acc_ptq = evaluate_tflite("model_ptq_int8.tflite", x_test, y_test)
acc_qat = evaluate_tflite("model_qat_int8.tflite", x_test, y_test)

print(f"PTQ INT8 accuracy : {acc_ptq:.4f}")
print(f"QAT INT8 accuracy : {acc_qat:.4f}")

# Export as C Arrays

The `.tflite` binary is embedded in a C header as a `const` byte array. The MCU firmware reads this array directly from flash.

Good practices for MCU C headers:
- `#ifndef` / `#define` / `#endif` — prevent double-inclusion
- `alignas(16)` — ensures 16-byte alignment for SIMD-like hardware (e.g. DSP cores)
- `const` — allows the linker to place the array in read-only flash

In [ ]:
import binascii
import datetime

def to_c_array(data: bytes) -> str:
    """Convert bytes to a comma-separated C hex array, 12 bytes per line."""
    hexstr = binascii.hexlify(data).decode("ascii").upper()
    values = ["0x" + hexstr[i:i+2] for i in range(0, len(hexstr), 2)]
    lines  = [", ".join(values[i:i+12]) for i in range(0, len(values), 12)]
    return ",\n  ".join(lines)

def write_c_header(tflite_path: str, header_path: str, array_name: str, quant_type: str):
    data = open(tflite_path, "rb").read()
    guard = array_name.upper() + "_H"
    body = (
        f"// Auto-generated on {datetime.date.today()}\n"
        f"// Model: {tflite_path}  |  Quantization: {quant_type}  |  Size: {len(data)} bytes\n"
        f"#ifndef {guard}\n"
        f"#define {guard}\n"
        f"#include <stdint.h>\n"
        f"#include <stdalign.h>\n\n"
        f"alignas(16) const uint8_t {array_name}[] = {{\n  "
        + to_c_array(data) +
        f"\n}};\n"
        f"const unsigned int {array_name}_len = {len(data)};\n\n"
        f"#endif  // {guard}\n"
    )
    open(header_path, "w").write(body)
    print(f"Written {header_path}  ({len(data)} bytes)")

write_c_header("model_ptq_int8.tflite", "model_ptq_int8.h", "tf_model_ptq", "INT8 PTQ")
write_c_header("model_qat_int8.tflite", "model_qat_int8.h", "tf_model_qat", "INT8 QAT")

# Deploying on a Microcontroller

Download `model_ptq_int8.h` (or `model_qat_int8.h`) from the **Files** panel and include it in your C++ project:

```cpp
#include "model_ptq_int8.h"   // provides tf_model_ptq[] and tf_model_ptq_len

// Pass the array to the LiteRT interpreter:
const tflite::Model* model = tflite::GetModel(tf_model_ptq);
```

### PTQ vs QAT — when to use which

| | Post-Training Quantization | Quantization-Aware Training |
|---|---|---|
| **Speed** | Fast — no extra training | Slower — requires fine-tuning |
| **Accuracy** | Excellent for simple models | Best when PTQ drops accuracy |
| **Complexity** | Low | Medium |

### Why INT8 matters on MCUs
- **~4× smaller** model — fits in limited flash (e.g. 256 KB on Arduino Nano 33 BLE)
- **~4× faster** inference — INT8 MACs are much cheaper than float32 on MCUs without FPUs
- **Lower power** consumption — fewer memory accesses

The [LiteRT for Microcontrollers](https://ai.google.dev/edge/litert/microcontrollers/overview) runtime (successor to TensorFlow Lite Micro) supports the generated `.tflite` files directly.

# Assignment: Inspect INT8 Quantization Parameters

Parse the exported `model_ptq_int8.tflite` flatbuffer to determine the **precision** and **granularity** of each quantized tensor type.

> Hint: use the `ai_edge_litert` interpreter's `get_tensor_details()` to inspect every tensor's `dtype` and `quantization_parameters`.


In [ ]:
import re
import numpy as np
from ai_edge_litert import interpreter as litert

# -- 1. Recover .tflite bytes from the embedded C header ---------------------
# The C header is in ../src/ relative to this notebook (train/)
HEADER_PATH = "../src/model_ptq_int8.h"

with open(HEADER_PATH) as f:
    src = f.read()

hex_values  = re.findall(r"0x([0-9A-Fa-f]{2})", src)
model_bytes = bytes(int(h, 16) for h in hex_values)
print(f"Recovered {len(model_bytes):,} bytes from header\n")

# -- 2. Load the flatbuffer into the LiteRT interpreter ----------------------
interp = litert.Interpreter(model_content=model_bytes)
interp.allocate_tensors()

input_idx  = {d["index"] for d in interp.get_input_details()}
output_idx = {d["index"] for d in interp.get_output_details()}

# -- 3. Print every tensor's dtype and quantization granularity --------------
print(f"{'idx':>4}  {'name':<40}  {'dtype':<8}  {'#scales':>8}  {'granularity'}")
print("-" * 80)

for t in interp.get_tensor_details():
    idx   = t["index"]
    name  = t["name"] or "<unnamed>"
    dtype = np.dtype(t["dtype"]).name
    qp    = t.get("quantization_parameters") or {}
    scales = qp.get("scales", np.array([]))
    n_sc  = len(scales)

    if n_sc == 0:
        gran = "no quantization"
    elif n_sc == 1:
        gran = "per-tensor"
    else:
        gran = f"per-channel  ({n_sc} scales)"

    role = ""
    if   idx in input_idx:  role = "  <- INPUT"
    elif idx in output_idx: role = "  -> OUTPUT"

    print(f"{idx:>4}  {name:<40}  {dtype:<8}  {n_sc:>8}  {gran}{role}")
